# Module 4, Section 3: Deep Agents 201: Orchestration and Verification

Section 2 built the TechHub support harness. Now we add three more levers:

1. **Dynamic subagents**: the agent writes a short script that dispatches subagents (loops, branches, parallel batches) instead of calling them one tool call at a time.
2. **Verification loops**: a grader checks the answer against a rubric and sends feedback until it passes or hits an iteration cap.

3. **Multi-model harnesses**: a large model plans, a smaller one executes, and a decision model (Jev) routes tickets and screens tool calls.

We finish with the **four loops of loop engineering** and where each one lives in this workshop.

## Setup

In [ ]:
import uuid
import warnings

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
warnings.filterwarnings("ignore", category=UserWarning)

from utils.langsmith_links import project_url, trace_link

print(f"📊 All traces from this workshop: {project_url()}")

In [ ]:
def new_thread():
    """A fresh conversation (thread) for each example."""
    return {"configurable": {"thread_id": str(uuid.uuid4())}}


def tool_calls(result):
    """Names of the tools the main agent called (subagent name in brackets for `task`)."""
    names = []
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", None) or []:
            extra = f"[{tc['args'].get('subagent_type')}]" if tc["name"] == "task" else ""
            names.append(tc["name"] + extra)
    return names


def truncate(text, limit=600):
    return text if len(text) <= limit else text[:limit] + "…"


def show(result, limit=600):
    """Print this turn's messages: human / ai / tool, as in a LangGraph trace."""
    messages = result["messages"]
    last_human = max((i for i, m in enumerate(messages) if m.type == "human"), default=0)
    for m in messages[last_human:]:
        m.model_copy(update={"content": truncate(m.text, limit)}).pretty_print()

## 1. The support inbox

We reuse the 12 questions from the Module 2 evaluation dataset as a batch of incoming support tickets. Several contain the customer's email; with PII redaction on, the agent sees `[REDACTED_EMAIL]`, and no customer is verified for a batch job.

In [ ]:
import json

tickets = [ex["inputs"]["question"] for ex in json.load(open("../module_2/baseline_dataset.json"))]
inbox = "\n".join(f"{i}. {q}" for i, q in enumerate(tickets, 1))
print(inbox)

## 2. Dynamic subagents

Dynamic subagents need two things: subagents to dispatch to, and the **code interpreter** middleware (`CodeInterpreterMiddleware`, from the `langchain-quickjs` package). The agent gets an `eval` tool that runs JavaScript, with a `task()` function that dispatches a subagent from code.

The word **"workflow"** in the request is the documented trigger for orchestrating through the interpreter.

In [ ]:
from agents import SupportContext, create_deep_support_agent

dynamic_agent = create_deep_support_agent(with_interpreter=True)

request = (
    "Run a workflow over this support inbox. For each ticket, classify it as account/order, product, or policy, "
    "dispatch it to the right specialist, and return one line per ticket: number, category, and a one-sentence "
    "resolution or 'needs identity verification'.\n\n" + inbox
)

with trace_link("Dynamic subagents over the inbox"):
    result = dynamic_agent.invoke(
        {"messages": [{"role": "user", "content": request}]},
        config=new_thread(),
        context=SupportContext(customer_id=None),  # batch job: no verified customer
    )

print("Main agent tool calls:", tool_calls(result))
print("Messages in the main context:", len(result["messages"]))
print("\n" + result["messages"][-1].text)

In [ ]:
# The orchestration code the agent wrote
for m in result["messages"]:
    for tc in getattr(m, "tool_calls", None) or []:
        if tc["name"] == "eval":
            print(tc["args"]["code"][:2500])
            break

The model wrote the dispatch loop once, and the interpreter ran it. Every ticket was covered because coverage is now a loop, not a series of model decisions.

Common orchestration patterns the agent can write (see the [dynamic subagents docs](https://docs.langchain.com/oss/python/deepagents/dynamic-subagents)):

| Pattern | Shape | Use when |
|---|---|---|
| Classify and act | Route each item to a specialist by type | Mixed inputs (like this inbox) |
| Fan out and synthesize | Same work over many items, then combine | Independent units, one report |
| Adversarial verification | Find, then independently verify | False positives are costly |
| Generate and filter | Several candidates, score, keep the best | Exploring options beats one shot |
| Tournament | Head-to-head judging, winners advance | Subjective criteria |
| Loop until done | Repeat passes until nothing new appears | Unknown scope |

## 3. Verification loop with `RubricMiddleware`

`RubricMiddleware` adds a grader subagent. After the agent answers, the grader scores the reply against a rubric and returns per-criterion feedback. The agent revises until the grader is satisfied or `max_iterations` is reached.

Two details matter in practice:
- The middleware only activates when you pass a **`rubric` in the invocation state**, so it's safe to leave in the stack.
- Each iteration costs a grader call plus a revision. Use it where quality matters more than latency. `RubricMiddleware` is in beta.

In [ ]:
from agents.deep_support_agent import DEFAULT_RUBRIC

print(DEFAULT_RUBRIC)

In [ ]:
evaluations = []
verified_agent = create_deep_support_agent(with_verifier=True, on_evaluation=evaluations.append)

with trace_link("Rubric verification loop"):
    result = verified_agent.invoke(
        {"messages": [{"role": "user", "content": "Can I return the laptop I opened last week?"}], "rubric": DEFAULT_RUBRIC},
        config=new_thread(),
        context=SupportContext(customer_id="CUST-003"),
    )

for e in evaluations:
    get = e.get if isinstance(e, dict) else lambda k: getattr(e, k)
    print(f"Iteration {get('iteration')}: {get('result')}")
    print(f"   {get('explanation')[:300]}\n")

print("Final answer:\n" + result["messages"][-1].text)

## 4. Multi-model harnesses

The model is part of the harness too, and it doesn't have to be one model. The main agent plans, delegates, and writes the reply; the specialists do narrow lookups with clear instructions. So `create_deep_support_agent` runs the specialists on `DEEP_AGENT_SMALL_MODEL` (`gpt-5.4-mini` with OpenAI, `claude-haiku-4-5` with Anthropic) and the main agent on `DEEP_AGENT_MODEL`. A subagent spec takes its own `model`:

```python
{**ORDER_SPECIALIST, "model": init_chat_model(DEEP_AGENT_SMALL_MODEL), "middleware": [...]}
```

Measure it rather than assume it. Run the same tickets through a single-model and a multi-model harness and compare tokens per model and wall time:

In [ ]:
import time

from langchain_core.callbacks import get_usage_metadata_callback

from config import DEEP_AGENT_MODEL, DEEP_AGENT_SMALL_MODEL

sample = [
    "What's the status of my order ORD-2025-0050?",
    "Does the MacBook Air M2 support two external monitors?",
    "Can I return a keyboard after 20 days if I opened it?",
]

harnesses = {
    "single-model": create_deep_support_agent(subagent_model=DEEP_AGENT_MODEL),
    "multi-model": create_deep_support_agent(subagent_model=DEEP_AGENT_SMALL_MODEL),
}
for label, harness in harnesses.items():
    start = time.perf_counter()
    with get_usage_metadata_callback() as usage, trace_link(f"Multi-model comparison: {label}"):
        for question in sample:
            harness.invoke(
                {"messages": [{"role": "user", "content": question}]},
                config=new_thread(),
                context=SupportContext(customer_id="CUST-003"),
            )
    print(f"\n{label}: {time.perf_counter() - start:.0f}s")
    for model_name, u in usage.usage_metadata.items():
        print(f"   {model_name}: {u['input_tokens']:,} in / {u['output_tokens']:,} out")

Whether the small model is good enough is an eval question, not a guess: run the Module 2 dataset against both harnesses and compare correctness before switching (Module 5 covers this in depth).

### Decision models: Jev as a routing step

Some harness steps aren't generation at all: *which specialist?*, *does this need identity verification?*, *how urgent is it?* A **decision model** such as TypeSafe's **Jev** answers typed questions with probabilities instead of text ([Building a harness with Jev](https://www.langchain.com/blog/building-a-harness-with-jev)):

| Question type | Answer | Read it as |
|---|---|---|
| `Noul` | Probability of "yes" | `r.nouls["needs_verification"].noul` → `0.89` |
| `Choice` | One option, with per-option probabilities and a confidence | `r.choices["category"].choice` → `"order"` |
| `Score` | An expected position on an ordered scale, from 0 to (levels − 1) | `r.scores["urgency"].score` → `1.15` on 0–2 |

`langchain-typesafe`'s `TypeSafeClassifier` is a LangChain Runnable, so every Jev call is traced in LangSmith next to the agent's runs. `utils/jev.py` builds one that goes through the LangSmith LLM Gateway, which holds the TypeSafe key: you authenticate with a LangSmith key and use a `typesafe/` model id, pinned because thresholds don't transfer between versions.

> ⚠️ **Which key:** `utils/jev.py` uses your `LANGSMITH_API_KEY`. The Gateway reads provider secrets from that key's workspace, so a workspace admin must first add a `TYPESAFE_API_KEY` provider secret there (otherwise: 424). If the secret is in another workspace you can access, set `JEV_WORKSPACE_ID` in `.env`.
>
> **Which data:** TypeSafe offers zero data retention per organization on request, and no BAA. Use Jev with synthetic or test data like TechHub, not customer data, until your security team has reviewed it.

Two habits from the [decision-model guide](https://docs.langchain.com/langsmith/llm-gateway-decision-models) matter most. Put the meaning of each option in `criteria`, because wording drives accuracy more than any threshold. And ask every question about one ticket in one request: Jev answers them in parallel.

Triage the inbox from §1 with Jev:

In [ ]:
from langchain_typesafe import Choice, Noul, Score

from utils.jev import jev_classifier

TRIAGE = {
    "category": Choice(
        instructions="Which team should handle this support ticket?",
        criteria={
            "order": "A specific order, delivery, refund, or the customer's account.",
            "product": "Product specs, compatibility, stock, or price.",
            "policy": "Store rules in general: returns, warranty, or shipping.",
        },
    ),
    "needs_verification": Noul(instructions="Does answering require looking up this customer's own orders or account?"),
    "urgency": Score(instructions="How urgent is this ticket?", criteria=["not urgent", "somewhat urgent", "very urgent"]),
}

start = time.perf_counter()
try:
    jev = jev_classifier().batch([{"state": t, "questions": TRIAGE} for t in tickets], config={"max_concurrency": 6})
except Exception as e:  # e.g. 424: the workspace has no TYPESAFE_API_KEY provider secret (see the note above)
    jev = None
    print(f"⏭️  Jev unavailable ({type(e).__name__}: {str(e)[:120]}). The Jev cells below are skipped.")
jev_seconds = time.perf_counter() - start

if jev:
    print(f"{jev[0].model}: {len(tickets)} tickets in {jev_seconds:.1f}s\n")
    for i, (ticket, r) in enumerate(zip(tickets, jev), 1):
        cat = r.choices["category"]
        print(f"{i:>2}. {cat.choice:<8} (conf={cat.confidence:.2f})  verify={r.nouls['needs_verification'].noul:.2f}  "
              f"urgency={r.scores['urgency'].score:.2f}/2  | {ticket[:55]}")

Is it right? Compare it with the small LLM doing the same job through structured output. `method="json_schema"` keeps the comparison fair: the default tool-calling method adds a tool schema to every request. This is the same "measure, don't assume" step as the harness comparison above, and the start of an eval (Module 5 goes further):

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field


class Triage(BaseModel):
    category: Literal["order", "product", "policy"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in TRIAGE["category"].criteria.items())
    )
    needs_verification: bool = Field(description=TRIAGE["needs_verification"].instructions)


llm_triage = init_chat_model(DEEP_AGENT_SMALL_MODEL).with_structured_output(Triage, method="json_schema")
start = time.perf_counter()
llm_out = llm_triage.batch(tickets, config={"max_concurrency": 6})
llm_seconds = time.perf_counter() - start

if jev:
    agree = sum(r.choices["category"].choice == o.category for r, o in zip(jev, llm_out))
    print(f"Jev: {jev_seconds:.1f}s | {DEEP_AGENT_SMALL_MODEL}: {llm_seconds:.1f}s | category agreement: {agree}/{len(tickets)}")
else:
    print(f"{DEEP_AGENT_SMALL_MODEL} triaged {len(tickets)} tickets in {llm_seconds:.1f}s (Jev skipped)")

The probabilities are the useful part for a harness: act on confident decisions and escalate the rest. A **confidence cascade** sends low-confidence tickets to the full agent and handles the rest with a fixed route. A `Choice` confidence is safe to gate on. A `Score` confidence measures how concentrated the distribution is, so a ticket sitting between two levels reads as low confidence even when Jev is clear about it: gate on the score's band instead.

In [ ]:
if not jev:
    print("⏭️  Skipped: Jev unavailable.")
else:
    THRESHOLD = 0.7  # tuned for the pinned Jev version; re-check it when the model changes

    for i, r in enumerate(jev, 1):
        cat = r.choices["category"]
        verify = r.nouls["needs_verification"].noul > 0.5
        if cat.confidence < THRESHOLD:
            route = "escalate to the main agent (ambiguous)"
        elif verify:
            route = "order-specialist, after identity verification"
        else:
            route = f"{'order' if cat.choice == 'order' else 'product-policy'}-specialist"
        urgent = " [priority]" if r.scores["urgency"].score >= 1.5 else ""  # a band on the 0-2 scale, never ==
        print(f"{i:>2}. {route}{urgent}")

### Jev inside the agent: a tool-safety gate

The same model can screen the agent's own actions. `JevToolGate` (`agents/middleware/jev_tool_gate.py`, the pattern from the official [langchain-typesafe demos](https://github.com/langchain-samples/langchain-typesafe-demos)) asks Jev one `Noul` about each tool call: *would this change data, move money, or affect someone outside the conversation?* Calls above 0.5 pause for a person with the same approve/reject payload as `interrupt_on`.

It complements deterministic controls; it doesn't replace them:
- `request_refund` stays in `interrupt_on` and always pauses. The gate skips it, and `task`, and covers the long tail: file writes, deletes, and any tool you add later.
- If Jev can't be reached, the gate **fails closed** and asks a person.
- Jev scores the calls once, right after the model proposes them, and the scores are kept in state (the way `HumanInTheLoopMiddleware` records edits). When a paused run resumes, the gate reads the stored score instead of asking Jev again, so a reviewer's "reject" can't be undone by a second, lower score.
- The customer-scope guard runs before the gate, so calls it refuses never reach a reviewer. `interrupt_on` works differently: it pauses right after the model proposes the call, before any tool middleware, so a reviewer can be asked about a refund the guard would refuse. The guard still refuses it when it runs.

First, how it scores some representative calls:

In [ ]:
from agents.middleware import JevToolGate

if not jev:
    print("⏭️  Skipped: Jev unavailable.")
else:
    gate = JevToolGate()
    for name, args in [
        ("get_order_status", {"order_id": "ORD-2025-0050"}),
        ("read_file", {"file_path": "/harness/AGENTS.md"}),
        ("write_file", {"file_path": "/memories/preferences.md", "content": "Prefers email updates."}),
        ("delete", {"file_path": "/memories/preferences.md"}),
        ("request_refund", {"order_id": "ORD-2024-0032", "reason": "dead pixels"}),
    ]:
        s = gate.score(name, args)
        print(f"{name:18} side effect p={s:.2f} -> {'asks a person' if s > gate.threshold else 'runs'}")

Now the agent with the gate on. A read-only question runs straight through; saving a preference writes to `/memories/`, which the gate pauses for approval:

In [ ]:
from langgraph.types import Command

if not jev:
    print("⏭️  Skipped: Jev unavailable.")
else:
    gated = create_deep_support_agent(jev_tool_gate=True)
    config, customer = new_thread(), SupportContext(customer_id="CUST-003")
    for question in ["What's the status of my order ORD-2025-0050?", "Please remember for next time that I prefer email updates."]:
        with trace_link(f"Jev tool gate: {question[:40]}"):
            r = gated.invoke({"messages": [{"role": "user", "content": question}]}, config=config, context=customer)
            while "__interrupt__" in r:
                # Parallel tool calls pause separately, so answer each interrupt by its id
                for pending in r["__interrupt__"]:
                    request = pending.value["action_requests"][0]
                    print(f"⏸️  Paused: {request['name']} ({request['description']}) -> approving")
                approve = {"decisions": [{"type": "approve"}]}
                r = gated.invoke(Command(resume={p.id: approve for p in r["__interrupt__"]}), config=config, context=customer)
        print(f"🧑 {question}\n🤖 {r['messages'][-1].text[:200]}\n")

In our runs the memory write scored just above 0.5 (0.54 to 0.58): that's the threshold lesson again. Pick the threshold from labelled examples of your own tool calls, and re-check it when the Jev version changes. Each gate decision is a `TypeSafeClassifier` run in the trace, so reviewers can see why a call paused.

`langchain-typesafe` also ships `ModelRouterMiddleware` and `AutoModeMiddleware` under `langchain_typesafe.experimental.middleware`. They're experimental, and the package is alpha (pinned to `0.0.1a3`). Jev can also act as a judge in LangSmith evals, which Module 5 covers.

## 5. Loop engineering: the four loops

The agent loop is only the first loop. Reliability and improvement come from the loops built around it ([The Art of Loop Engineering](https://www.langchain.com/blog/the-art-of-loop-engineering)):

| Loop | What it does | Primitive | In this workshop |
|---|---|---|---|
| 1. Agent loop | Model calls tools until done | `create_agent` / `create_deep_agent` | Modules 1 and 4 |
| 2. Verification loop | A grader checks output against a rubric | `RubricMiddleware`, offline evals | This section; Module 5 |
| 3. Event-driven loop | Schedules, webhooks, or channels trigger runs | LangSmith Deployment crons, MDA schedules | Module 6 |
| 4. Hill-climbing loop | Production traces drive harness improvements | Online evals, LangSmith Engine | Module 3 (online evals, the data flywheel); Engine is covered in the Monitoring and Governance workshop |

**Execution environment is a harness choice too.** This module runs tools in-process. Module 6 moves the agent to a managed runtime and gives it a sandbox to run code safely.

## Recap

- Dynamic subagents turn orchestration into code: coverage is guaranteed by a loop, and the main context stays small.
- `RubricMiddleware` adds runtime verification. Pass a `rubric` to switch it on, and weigh the extra latency.
- Multi-model harnesses put the big model where judgment matters and a small model where the task is narrow. Measure the trade-off with evals.
- A decision model like Jev returns typed answers with probabilities. Use the confidence to route (act on confident decisions, escalate the rest), and as a tool-safety gate that sends likely side effects to a person. `TypeSafeClassifier` traces every decision in LangSmith.
- Harness engineering is iterative: read the trace, change one lever, re-verify.